# Assignment 5 – FastAPI & REST API

## Question 1: What is a REST API?

REST (Representational State Transfer) is an architectural style for designing networked applications. A REST API is an interface that allows different software systems to communicate over HTTP using standard methods. REST APIs are stateless — each request contains all the information needed to process it — and use standard HTTP verbs to perform operations on resources identified by URLs.

### HTTP Methods

- **GET:** Retrieves data from the server. It is read-only and idempotent (calling it multiple times gives the same result). Example: `GET /courses` retrieves a list of courses.
- **POST:** Sends new data to the server to create a new resource. It is NOT idempotent — calling it multiple times creates multiple records. Example: `POST /courses` creates a new course.
- **PUT:** Updates an existing resource by replacing it entirely with the new data provided. It is idempotent. Example: `PUT /courses/1` updates course with ID 1.
- **DELETE:** Removes an existing resource from the server. It is idempotent. Example: `DELETE /courses/1` deletes course with ID 1.

### Why FastAPI is Preferred Over Flask for Modern APIs

- **Performance:** FastAPI is built on Starlette and is one of the fastest Python frameworks, comparable to Node.js and Go. Flask is synchronous and slower for high-concurrency scenarios.
- **Async Support:** FastAPI natively supports async/await, allowing non-blocking I/O operations. Flask requires extensions for async support.
- **Automatic Validation:** FastAPI uses Pydantic for request/response validation automatically. Flask requires manual validation or third-party libraries like Marshmallow.
- **Auto-generated Docs:** FastAPI automatically generates interactive Swagger UI and ReDoc documentation. Flask has no built-in documentation generation.
- **Type Hints:** FastAPI leverages Python type hints for validation, serialization, and editor support. Flask doesn't use type hints natively.
- **Dependency Injection:** FastAPI has a powerful built-in dependency injection system. Flask has no built-in DI system.

## Question 2: FastAPI Application with Endpoints

Below is a FastAPI application implementing all required endpoints:

In [ ]:
from fastapi import FastAPI
from typing import Optional

app = FastAPI()

courses = [
    {"id": 1, "title": "Python Basics", "level": "beginner"},
    {"id": 2, "title": "FastAPI Advanced", "level": "advanced"},
    {"id": 3, "title": "Data Science", "level": "intermediate"},
]

# Root endpoint
@app.get('/')
def root():
    return {"message": "Welcome to the Course API"}

# /courses GET endpoint
@app.get('/courses')
def get_courses(level: Optional[str] = None):
    if level:
        return [c for c in courses if c['level'] == level]
    return courses

# /courses/{course_id} path parameter
@app.get('/courses/{course_id}')
def get_course(course_id: int, level: Optional[str] = None):
    course = next((c for c in courses if c['id'] == course_id), None)
    if not course:
        return {"error": "Course not found"}
    return course

### Path Parameter vs Query Parameter

- **Path Parameter:** Part of the URL path itself. Used to identify a specific resource. Example: `/courses/1` — here `1` is the path parameter `course_id`. It is required and cannot be omitted.
- **Query Parameter:** Appended to the URL after a `?` and used for filtering, sorting, or optional configuration. Example: `/courses?level=beginner` — here `level` is a query parameter. It is typically optional.

## Question 3: Validation with FastAPI

FastAPI uses `Path()`, `Query()`, and Pydantic's `Field()` for built-in validation:

In [ ]:
from fastapi import FastAPI, Path, Query, HTTPException
from typing import Optional
from enum import Enum

class Level(str, Enum):
    beginner = 'beginner'
    intermediate = 'intermediate'
    advanced = 'advanced'

@app.get('/courses/{course_id}')
def get_course(
    course_id: int = Path(..., gt=0, description='Must be integer > 0'),
    level: Optional[Level] = Query(None)
):
    return {'course_id': course_id, 'level': level}

### What Happens When Validation Fails

When validation fails, FastAPI automatically returns a **422 Unprocessable Entity** HTTP response. It does NOT raise a 500 server error or crash. The response includes a detailed JSON body indicating exactly which field failed, what the error type is, and where in the request the error occurred (e.g., path, query, body). This helps clients immediately understand what went wrong without any extra code from the developer.

## Question 4: Pydantic Model with Validation

In [ ]:
from pydantic import BaseModel, Field, validator
from typing import Optional

class Course(BaseModel):
    title: str
    price: float = Field(..., gt=0, description='Price must be greater than 0')
    duration: int = Field(..., gt=0, description='Duration must be greater than 0')
    published: bool = False  # Default is False

    @validator('price')
    def price_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError('Price must be a positive number')
        return v

### Why Schema Validation is Critical in APIs

- **Data Integrity:** Ensures only valid, expected data enters the system. Without validation, malformed data can corrupt databases or cause unexpected behavior.
- **Security:** Prevents injection attacks and malicious payloads. Strict type checking means attackers can't pass unexpected data types.
- **Clear Error Messages:** Validated APIs return descriptive errors to clients (e.g., `'price must be > 0'`), improving developer experience.
- **Automatic Documentation:** Pydantic models integrate with FastAPI to auto-generate accurate API docs with correct types and constraints.
- **Reliability:** Validation acts as a contract between client and server, reducing bugs caused by type mismatches or missing fields.

## Question 5: Authentication vs Authorization & JWT

### Authentication vs Authorization

- **Authentication:** The process of verifying WHO a user is. It answers: *"Are you who you claim to be?"* Examples: logging in with a username/password, verifying a JWT token. Authentication happens first.
- **Authorization:** The process of determining WHAT an authenticated user is allowed to do. It answers: *"Do you have permission to access this resource?"* Examples: admin can delete users, regular user cannot. Authorization happens after authentication.

### JWT-Based Authentication

JWT (JSON Web Token) is a compact, self-contained token format for securely transmitting information between parties. A JWT consists of three base64-encoded parts separated by dots:

- **Header:** Contains the algorithm (e.g., HS256) and token type.
- **Payload:** Contains claims (user data like `user_id`, `role`, expiry time).
- **Signature:** Ensures the token has not been tampered with.

**Flow:** User logs in → server validates credentials → server creates and signs a JWT → client stores the JWT → client sends JWT in the `Authorization` header on every request → server verifies the signature and grants access.

### Why Stateless Authentication is Scalable

In stateless authentication (like JWT), the server does NOT store session data. All needed information is encoded in the token itself. This is scalable because:

- Any server instance can verify the token independently — no need for a shared session database.
- Horizontal scaling is easy — add more servers without synchronizing session state.
- Reduces database load — no session lookups needed on every request.
- Works well with microservices and distributed systems.

## Question 6: API Testing

### Why API Testing is Critical

API testing ensures that the API behaves correctly, returns expected responses, handles errors gracefully, and maintains contracts with clients. It catches regressions early, validates business logic, ensures security rules are enforced, and gives confidence when refactoring or deploying new features.

### Unit Testing vs Integration Testing

- **Unit Testing:** Tests a single function or component in isolation. External dependencies (databases, APIs) are mocked. Fast to run. Example: testing a function that calculates the discounted price of a course.
- **Integration Testing:** Tests multiple components working together including real dependencies like databases or HTTP endpoints. Slower but more realistic. Example: testing that `POST /courses` actually saves a course to the database and returns 201.

### Sample Test Case Using FastAPI TestClient

In [ ]:
from fastapi.testclient import TestClient
from main import app  # import your FastAPI app

client = TestClient(app)

def test_root():
    response = client.get('/')
    assert response.status_code == 200
    assert response.json() == {"message": "Welcome to the Course API"}

def test_get_courses():
    response = client.get('/courses')
    assert response.status_code == 200
    assert isinstance(response.json(), list)

def test_invalid_course_id():
    response = client.get('/courses/-1')
    assert response.status_code == 422  # Validation error

## Question 7: Python Type Hints in FastAPI

- **`Optional`:** Used when a value can be either a specific type OR `None`. Useful for optional query parameters or nullable fields. Example: `Optional[str]` means the value can be a string or None.
- **`List[str]`:** Specifies a list where every element must be a string. FastAPI automatically validates that each item in the list is a string. Example: `List[str]` for a list of course tags.
- **`Union`:** Allows a value to be one of multiple types. Example: `Union[int, str]` means the value can be an integer or a string. In Python 3.10+, this can be written as `int | str`.
- **`Annotated`:** Used to attach metadata or validation constraints to a type. FastAPI uses `Annotated` to combine type hints with `Field()` or `Path()` or `Query()` validators. Example: `Annotated[int, Field(gt=0)]` means an integer greater than 0.

### Why Proper Typing Improves Maintainability

- **Self-documenting code:** Types serve as inline documentation — developers immediately understand what a function expects and returns.
- **IDE support:** Type hints enable autocomplete, type checking, and refactoring tools in editors like VSCode and PyCharm.
- **Early error detection:** Static analysis tools like `mypy` can catch type errors before runtime.
- **Automatic validation:** FastAPI uses type hints to validate incoming requests automatically.
- **Easier onboarding:** New developers can understand codebase faster when types are explicit.

## Question 8: Path Parameter vs Query Parameter vs Request Body

- **Path Parameter:** Embedded directly in the URL path. Identifies a specific resource uniquely. Always required. Example: `/courses/5` — `5` is the course ID. Use when: the parameter is essential to identify the resource (IDs, slugs).
- **Query Parameter:** Added after `?` in the URL. Used for optional filtering, sorting, or configuration. Example: `/courses?level=beginner&page=2`. Use when: the parameter is optional or used for filtering/searching/pagination.
- **Request Body:** Data sent in the HTTP request body (typically JSON). Used for creating or updating resources with complex data. Example: `POST /courses` with body `{"title": "FastAPI", "price": 999}`. Use when: sending structured data to create or modify a resource (POST, PUT, PATCH).

### When to Use Each

- **Path parameter** → `GET /users/42` (fetch user with ID 42)
- **Query parameter** → `GET /products?category=books&sort=price`
- **Request body** → `POST /orders` with full order details in JSON

## Question 9: BaseModel, Field, validator, root_validator

- **`BaseModel`:** The foundation class from Pydantic. All data models inherit from it. It handles automatic parsing, validation, and serialization of data based on field type annotations. Example: `class Course(BaseModel): title: str`
- **`Field`:** A function used inside a `BaseModel` to add metadata and constraints to individual fields. Allows setting default values, minimum/maximum values, descriptions, aliases, etc. Example: `price: float = Field(..., gt=0, description='Must be positive')`
- **`validator`:** A decorator used to define custom validation logic for a single field. Runs after type coercion. Example: `@validator('email') def validate_email(cls, v): if '@' not in v: raise ValueError('Invalid email')`
- **`root_validator`:** A decorator used to validate multiple fields together or cross-field validation. Runs after all individual field validators. Example: Checking that `end_date` is after `start_date` — this requires access to both fields simultaneously, which `root_validator` provides.

## Question 10: Pydantic BaseSettings for Configuration

Pydantic's `BaseSettings` class allows you to define application settings that are read from environment variables automatically:

In [ ]:
from pydantic import BaseSettings
from functools import lru_cache

class Settings(BaseSettings):
    database_url: str
    secret_key: str
    debug: bool = False

    class Config:
        env_file = ".env"

@lru_cache()
def get_settings():
    return Settings()

# Usage in FastAPI
from fastapi import Depends

@app.get('/info')
def info(settings: Settings = Depends(get_settings)):
    return {'debug': settings.debug}

### Why Secrets Should Never Be Hardcoded

- **Security Risk:** Hardcoded secrets in source code are exposed to anyone with access to the codebase, including via Git history even after deletion.
- **Version Control Exposure:** If code is pushed to GitHub (even a private repo), secrets are permanently in the commit history.
- **Environment Flexibility:** Different environments (dev, staging, production) need different secrets. Hardcoding forces code changes per environment.
- **Rotation Difficulty:** Rotating a hardcoded secret requires a code change and redeployment. Environment variables can be changed without touching code.
- **Compliance:** Standards like PCI-DSS, SOC2, and OWASP explicitly prohibit hardcoded credentials. Violations can result in legal or financial penalties.

**Best Practice:** Store secrets in environment variables or dedicated secret managers (AWS Secrets Manager, HashiCorp Vault) and load them via Pydantic `BaseSettings`.